In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

path = "/content/drive/MyDrive/RF_Dataset"
os.makedirs(path, exist_ok=True)

print("Folder created at:", path)

Folder created at: /content/drive/MyDrive/RF_Dataset


In [3]:
!wget -P /content/drive/MyDrive/RF_Dataset https://opendata.deepsig.ai/datasets/2016.10/RML2016.10a_dict.pkl

--2026-03-18 12:54:35--  https://opendata.deepsig.ai/datasets/2016.10/RML2016.10a_dict.pkl
Resolving opendata.deepsig.ai (opendata.deepsig.ai)... failed: Name or service not known.
wget: unable to resolve host address ‘opendata.deepsig.ai’


In [4]:
import os

files = os.listdir('/content/drive/MyDrive/RF_Dataset')
print(files)

[]


In [5]:
!wget https://github.com/radioML/dataset/raw/master/RML2016.10a_dict.pkl

--2026-03-18 13:17:17--  https://github.com/radioML/dataset/raw/master/RML2016.10a_dict.pkl
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-18 13:17:18 ERROR 404: Not Found.



In [6]:
import pickle

file_path = "RML2016.10a_dict.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Dataset loaded successfully")
print("Total keys:", len(data))

UnpicklingError: pickle data was truncated

In [8]:
import pickle

file_path = "/content/drive/MyDrive/RML2016.10a_dict.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Dataset loaded successfully")

Dataset loaded successfully


In [9]:
import os

print(os.path.getsize(file_path))

640919653


In [10]:
import numpy as np

X = []
y = []

limit = 50000   # use only 50k samples (safe for Colab)
count = 0

for key in data.keys():
    signals = data[key]
    label = key[0]

    for signal in signals:
        X.append(signal)
        y.append(label)
        count += 1

        if count >= limit:
            break
    if count >= limit:
        break

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 2, 128)
y shape: (50000,)


In [11]:
X = X.transpose(0, 2, 1)
print("Final shape:", X.shape)

Final shape: (50000, 128, 2)


In [12]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Classes:", encoder.classes_)

Classes: ['8PSK' 'AM-DSB' 'AM-SSB' 'BPSK' 'CPFSK' 'GFSK' 'PAM4' 'QAM16' 'QAM64'
 'QPSK' 'WBFM']


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (40000, 128, 2)
Test shape: (10000, 128, 2)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense

model = Sequential()

model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(128, 2)))
model.add(MaxPooling1D(pool_size=2))

model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dense(len(set(y_encoded)), activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 126, 64)        │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 61, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       491,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │         1,419 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 518,219 (1.98 MB)

 Trainable params: 518,219 (1.98 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.2071 - loss: 2.1112 - val_accuracy: 0.2432 - val_loss: 1.9864
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.2919 - loss: 1.8719 - val_accuracy: 0.2981 - val_loss: 1.8407
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.3501 - loss: 1.7565 - val_accuracy: 0.3731 - val_loss: 1.7150
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.3925 - loss: 1.6610 - val_accuracy: 0.3721 - val_loss: 1.6755
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4101 - loss: 1.6019 - val_accuracy: 0.4116 - val_loss: 1.6077
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4294 - loss: 1.5611 - val_accuracy: 0.4245 - val_loss: 1.5727
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4425 - loss: 1.5312 - val_accuracy: 0.4152 - val_loss: 1.5641
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4554 - loss: 1.4996 - val_accuracy: 0.

In [16]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.4579 - loss: 1.4938
Test Accuracy: 0.4578999876976013


In [1]:
# Example: treat some modulations as "authorized"
authorized_classes = ['BPSK', 'QPSK', '8PSK']

y_binary = []

for label in y:
    if label in authorized_classes:
        y_binary.append(0)   # Authorized
    else:
        y_binary.append(1)   # Rogue

y_binary = np.array(y_binary)

print("Binary labels created")

NameError: name 'y' is not defined

In [2]:
import numpy as np

X = []
y = []

limit = 50000
count = 0

for key in data.keys():
    signals = data[key]
    label = key[0]

    for signal in signals:
        X.append(signal)
        y.append(label)
        count += 1

        if count >= limit:
            break
    if count >= limit:
        break

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

NameError: name 'data' is not defined

In [3]:
import pickle

file_path = "/content/drive/MyDrive/RML2016.10a_dict.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Dataset loaded")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/RML2016.10a_dict.pkl'

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

print(os.listdir('/content/drive/MyDrive'))

['ABHA-Card-91-2702-2753-0283.pdf', '1727075244067.jpg', '1727112545258.jpg', 'Classroom', 'Colab Notebooks', 'Assignment 1.pdf', 'DocScanner 28 Jul 2025 5-20\u202fpm.pdf', 'Document from gokulsenthilganesh (1)', 'Gokul S Resume.docx', 'Document from gokulsenthilganesh (2).pdf', 'Photo from gokulsenthilganesh (5)', 'IMG_20250814_203407.jpg', 'IMG_20250814_203526.jpg', 'DocScanner 14 Aug 2025 8-41\u202fpm', 'DocScanner 14 Aug 2025 8-42\u202fpm', 'DocScanner 14 Aug 2025 8-44\u202fpm', 'IMG_20250814_205650.jpg', 'Photo from gokulsenthilganesh (4)', 'Photo from gokulsenthilganesh (3)', 'Photo from gokulsenthilganesh (2)', 'Photo from gokulsenthilganesh (1)', 'Photo from gokulsenthilganesh', 'Document from gokulsenthilganesh', 'Document from gokulsenthilganesh (1).pdf', 'Document from gokulsenthilganesh.pdf', 'Krithik_pdf.pdf', 'Doc Scanner', 'Doc Scanner Upload', 'car_ws.zip', 'yolo_custom_project', 'YOLO_Models', 'Document from gokul (1).pdf', '12f.pdf', '12.pdf', 'YOLO_Backup', 'YOLO_MOD

In [6]:
import pickle

file_path = "/content/drive/MyDrive/RML2016.10a_dict.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Dataset loaded successfully")
print("Total keys:", len(data))

Dataset loaded successfully
Total keys: 220


In [7]:
import numpy as np

X = []
y = []

limit = 50000
count = 0

for key in data.keys():
    signals = data[key]
    label = key[0]

    for signal in signals:
        X.append(signal)
        y.append(label)
        count += 1

        if count >= limit:
            break
    if count >= limit:
        break

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 2, 128)
y shape: (50000,)


In [8]:
authorized_classes = ['BPSK', 'QPSK', '8PSK']

y_binary = np.array([0 if lbl in authorized_classes else 1 for lbl in y])

# Check balance
print("Authorized (0):", (y_binary==0).sum(),
      "Rogue (1):", (y_binary==1).sum())

Authorized (0): 16000 Rogue (1): 34000


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (40000, 2, 128) Test: (10000, 2, 128)


In [10]:
# simple normalization
mean = X_train.mean()
std  = X_train.std() + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Dropout, Flatten, Dense

model = Sequential([
    Conv1D(64, 5, activation='relu', input_shape=(128, 2)),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(128, 3, activation='relu'),
    MaxPooling1D(2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')   # binary output
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 124, 64)        │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 124, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 62, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 60, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 28, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 14, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       229,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,089 (1.16 MB)

 Trainable params: 304,705 (1.16 MB)

 Non-trainable params: 384 (1.50 KB)

In [12]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'conv1d' is incompatible with the layer: expected axis -1 of input shape to have value 2, but received input with shape (64, 2, 128)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(64, 2, 128), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [13]:
X_train = X_train.transpose(0, 2, 1)
X_test  = X_test.transpose(0, 2, 1)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (40000, 128, 2)
Test shape: (10000, 128, 2)


In [14]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/30


ValueError: Creating variables on a non-first call to a function decorated with tf.function.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pickle

file_path = "/content/drive/MyDrive/RML2016.10a_dict.pkl"

with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

print("Dataset loaded successfully")

Dataset loaded successfully


In [3]:
import numpy as np

X = []
y = []

limit = 50000
count = 0

for key in data.keys():
    signals = data[key]
    label = key[0]

    for signal in signals:
        X.append(signal)
        y.append(label)
        count += 1

        if count >= limit:
            break
    if count >= limit:
        break

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (50000, 2, 128)
y shape: (50000,)


In [4]:
# Convert (N, 2, 128) → (N, 128, 2)
X = X.transpose(0, 2, 1)

print("Final shape:", X.shape)

Final shape: (50000, 128, 2)


In [5]:
authorized_classes = ['BPSK', 'QPSK', '8PSK']

y_binary = np.array([0 if lbl in authorized_classes else 1 for lbl in y])

print("Authorized:", (y_binary==0).sum())
print("Rogue:", (y_binary==1).sum())

Authorized: 16000
Rogue: 34000


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

In [7]:
mean = X_train.mean()
std = X_train.std() + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

In [8]:
import tensorflow as tf
tf.keras.backend.clear_session()

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Dropout, Flatten, Dense

model = Sequential([
    Conv1D(64, 5, activation='relu', input_shape=(128, 2)),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(128, 3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(2),

    Conv1D(128, 3, activation='relu'),
    MaxPooling1D(2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 124, 64)        │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 124, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 62, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 60, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 28, 128)        │        49,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 14, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       229,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 305,089 (1.16 MB)

 Trainable params: 304,705 (1.16 MB)

 Non-trainable params: 384 (1.50 KB)

In [10]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8394 - loss: 0.3210 - val_accuracy: 0.8379 - val_loss: 0.3254
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8456 - loss: 0.3006 - val_accuracy: 0.8458 - val_loss: 0.2966
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8497 - loss: 0.2867 - val_accuracy: 0.8469 - val_loss: 0.2928
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8550 - loss: 0.2717 - val_accuracy: 0.8426 - val_loss: 0.3053
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8585 - loss: 0.2669 - val_accuracy: 0.8446 - val_loss: 0.2967
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8641 - loss: 0.2515 - val_accuracy: 0.8458 - val_loss: 0.2967
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8762 - loss: 0.2383 - val_accuracy: 0.8425 - val_loss: 0.3315
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8884 - loss: 0.2184 - val_accuracy: 0.

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8227 - loss: 1.0489
Test Accuracy: 0.822700023651123
